# Setup and Load Data #

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import os
import itertools
from io import StringIO
import inspect

# Visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px

# Time Series & Stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Sklearn: Preprocessing, Modeling, Metrics
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, 
    TimeSeriesSplit, cross_val_score
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, 
    mean_absolute_percentage_error, r2_score, 
    accuracy_score, classification_report
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import (
    KMeans, AgglomerativeClustering
)
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Clustering & Distance
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform, cdist

# Geospatial
import folium

# HDBSCAN
import hdbscan

# ML Model
from xgboost import XGBRegressor

# Dash & JupyterDash
from dash import Dash, dcc, html, Input, Output, dash_table
from jupyter_dash import JupyterDash

# Typing
from typing import List, Tuple, Dict


In [ ]:
#set the working directory
import os
os.chdir('/Users/max/Satoshi test Python/Capstone/Dateien')
#os.chdir('/Users/hquangn/Downloads/OneDrive_1_15-07-2025')


In [ ]:
#Dataset
df= pd.read_csv('Capstone_Stand_10_04_2025_anonym.csv', sep = ';',header=0)
df2 = pd.read_csv('Capstone_Stand_18_06_2025_anonym.csv', sep = ';',header=0)
geo_df = pd.read_csv('georef-germany-postleitzahl.csv', sep=';', header=0)
kunde_branche = pd.read_csv('Kunde_Branche.csv', sep = ';',header=0)
kunde_geschaeftsart = pd.read_csv('Kunde_Geschaeftsart.csv', sep = ';',header=0)
kunde_sortiment = pd.read_csv('Kunde_Sortiment.csv', sep = ';',header=0)
rubrikengruppe = pd.read_csv('Rubrikengruppe.csv', sep = ';',header=0)

In [ ]:
#combine 2 sales dataset (1 from 2020-2024, 1 for 5 first months of 2025)
df= pd.concat([df, df2], ignore_index=True)

In [ ]:
#Create tables for matching codified numbers to names in original table
product_l1 = pd.DataFrame({
    'Titelgruppe': [1, 2],
    'Titelgruppe_name': ['ABO', 'Express']
})
product_l2 = pd.DataFrame({
    'Hauptausgabengruppe_aggregiert': [1, 2],
    'Hauptausgabengruppe_aggregiert_name': ['Print', 'Digital']
})
product_l3 = pd.DataFrame({
    'Hauptausgabengruppe_Bezeichnung': [1, 2, 3, 4, 5, 6],
    'Hauptausgabengruppe_Bezeichnung_name': ['Offline Service','Print-to-Online','Print','Extra Service','Flyer inserts','Digital']
})

business_type = pd.DataFrame({
    'Geschäftsart': [1, 2],
    'Geschäftsart_name':['B2B', 'B2C']
})

# Data Preparation and Cleaning #

##Change data formatting ##

In [ ]:
# Change datatype
# Jahr
df['Jahr'] = pd.to_numeric(df['Jahr'], errors='coerce').astype('Int64')
# Monat
# Convert German month names to month numbers (as string, e.g., '01', '02', ...)
df['Monat'] = df['Monat'].astype(str).str.strip()
monats_map = {
    'Januar': '01', 'Februar': '02', 'März': '03', 'April': '04',
    'Mai': '05', 'Juni': '06', 'Juli': '07', 'August': '08',
    'September': '09', 'Oktober': '10', 'November': '11', 'Dezember': '12'
}
df['Monat_num'] = df['Monat'].map(monats_map)
#drop rows with NaN in Monat_num
df = df.dropna(subset=['Monat_num'])
# Convert Monat_num to numeric
df['Monat_num'] = pd.to_numeric(df['Monat_num'], errors='coerce')

# Verkauf
df['Verkauf'] = df['Verkauf'].astype(str).str.replace(',', '.', regex=False)
df['Verkauf'] = pd.to_numeric(df['Verkauf'], errors='coerce')

# Jahr
df['Jahr'] = pd.to_numeric(df['Jahr'], errors='coerce')


# Combine Jahr and Monat_num into a datetime column
df['Jahr-Monat'] = pd.to_datetime(
    df['Jahr'].astype('Int64').astype(str) + '-' + df['Monat_num'].astype(str).str.zfill(2),
    format='%Y-%m',
    errors='coerce'
)
df['Jahr-Monat'] = df['Jahr-Monat'].dt.to_period('M')

# Add Quarter
df['Quarter'] = df['Jahr-Monat'].dt.quarter.astype('Int64')

###Adjust sales values by adding back the discount to its respective periods ##

In [ ]:
# under the assumption that the discount is appiled by equal weighted proportion respective to each sales value in the same period
#Create lookup tables for total discount and total positive sales per Jahr-Monat
discount_lookup = df[df['Verkauf'] < 0].groupby('Jahr-Monat')['Verkauf'].sum().abs()
sales_lookup = df[df['Verkauf'] > 0].groupby('Jahr-Monat')['Verkauf'].sum()
sales_lookup = df[df['Verkauf'] > 0].groupby('Jahr-Monat')['Verkauf'].sum()

# Map those values to the full dataframe
df['total_discount'] = df['Jahr-Monat'].map(discount_lookup)
df['total_sales'] = df['Jahr-Monat'].map(sales_lookup)

# Compute proportion of each positive sale within its Jahr-Monat (no rounding)
df['sale_share'] = df['Verkauf'] / df['total_sales']
df['sale_share'] = df['sale_share'].where(df['Verkauf'] > 0, np.nan)

# Apply proportional discount to positive sales (no rounding)
df['adjusted_sales'] = np.where(
    df['Verkauf'] > 0,
    df['Verkauf'] - (df['sale_share'] * df['total_discount']),
    np.nan
)

# Validate totals
total_positive_sales = df[df['Verkauf'] > 0].groupby('Jahr-Monat')['Verkauf'].sum()
total_adjusted_sales = df.groupby('Jahr-Monat')['adjusted_sales'].sum()
total_discount = df[df['Verkauf'] < 0].groupby('Jahr-Monat')['Verkauf'].sum().abs()

# Create final summary DataFrame and apply rounding only for display
df_total = pd.DataFrame({
    'total_positive_sales': total_positive_sales,
    'total_adjusted_sales': total_adjusted_sales,
    'total_discount': total_discount
})
df_total['diff'] = df_total['total_positive_sales'] - df_total['total_adjusted_sales']
df_total = df_total.round(2).reset_index()

# Drop columns if they exist
df.drop(columns=['total_discount', 'total_sales', 'sale_share'], inplace=True, errors='ignore')


In [ ]:
# Verkauf by Jahr
Verkauf_table = df.groupby('Jahr-Monat')[['Verkauf','adjusted_sales']].sum().reset_index()
display(Verkauf_table)

In [ ]:
df.info()

### Define Areas Cologne, West, East and National using Geo Data ###

In [ ]:
#Geo data
# Step 1: Create a new PLZ3 column from the 5-digit 'Name' column
geo_df['PLZ3'] = geo_df['Name'].astype(str).str[:3]

# Step 2: Split 'geo_point_2d' into separate latitude and longitude columns
geo_df[['latitude', 'longitude']] = geo_df['geo_point_2d'].str.split(',', expand=True).astype(float)

# Step 3: Drop duplicates to get one coordinate per PLZ3
geo_plz3_coords = geo_df.groupby('PLZ3')[['latitude', 'longitude']].mean().reset_index()

##Join geo data with df
# Step 1: Ensure data types match
geo_plz3_coords['PLZ3'] = geo_plz3_coords['PLZ3'].astype(str)

# Step 2: Merge coordinates into df
df = df.merge(geo_plz3_coords, left_on='Kunde_PLZ', right_on='PLZ3', how='left')

# Step 3: Drop duplicate PLZ3 column if needed
df.drop(columns=['PLZ3'], inplace=True)

# Step 4: Preview result
print(df[['Kunde_PLZ', 'latitude', 'longitude']].drop_duplicates().head())

In [ ]:
## Join 'Kreise' and 'Länder' into df
# Step 1: Ensure both dataframes have PLZ3
df['PLZ3'] = df['Kunde_PLZ'].astype(str).str[:3]
geo_df['PLZ3'] = geo_df['Name'].astype(str).str[:3]

# Step 2: Prepare a mapping of PLZ3 to one Kreis name (first occurrence)
plz3_kreis = geo_df[['PLZ3', 'Kreis name']].drop_duplicates(subset='PLZ3')

# Step 3: Merge into df
df = df.merge(plz3_kreis, on='PLZ3', how='left')


#Step 2: Prepare a mapping of PLZ3 to one Kreis name (first occurrence)
plz3_land = geo_df[['PLZ3', 'Land name']].drop_duplicates(subset='PLZ3')

# Step 3: Merge into df
df = df.merge(plz3_land, on='PLZ3', how='left')

In [ ]:
# Allocate regions based on Kreis names - KstA
def assign_region_from_kreis(kreis):
    if kreis == 'Kreisfreie Stadt Köln':
        return 'Cologne'
    elif kreis in [
        'Kreis Rhein-Erft-Kreis',
        'Kreis Euskirchen'   
    ]:
        return 'West'
    elif kreis in [
        'Kreis Oberbergischer Kreis',
        'Kreis Rhein-Sieg-Kreis',
        'Kreis Rheinisch-Bergischer Kreis',
        'Kreisfreie Stadt Bonn'
    ]:
        return 'East'
    else:
        return 'National/International'
    
# Apply function to assign new region_group based on Kreis name
df['area'] = df['Kreis name'].apply(assign_region_from_kreis)


## Handle Outliers and Missing Values ##

In [ ]:
#Handling missing values
#fill in Verkaufsprodukt_Bezeichnung and Produktionsprodukt_Bezeichnung with 999
df['Verkaufsprodukt_Bezeichnung'] = df['Verkaufsprodukt_Bezeichnung'].fillna(999)
df['Produktionsprodukt_Bezeichnung'] = df['Produktionsprodukt_Bezeichnung'].fillna(999)

#For only null values in Kunde_Branche, fill in Kunde_Branche with 13 if Geschäftsart ==1; 14 if Geschäftsart ==2
df['Kunde_Branche'] = df.apply(
    lambda x: 13 if pd.isna(x['Kunde_Branche']) and x['Geschäftsart'] == 1
    else 14 if pd.isna(x['Kunde_Branche']) and x['Geschäftsart'] == 2
    else x['Kunde_Branche'],
    axis=1
)

#fill in Rubrikengruppe with 99
df['Rubrikengruppe'] = df['Rubrikengruppe'].fillna(99)

#fill in  Kunde_Geschäftsart with 998 if Kunde_Branche == 13; 999 if Kunde_Branche == 14
df['Kunde_Geschäftsart']= df.apply(
    lambda x: 998 if x['Kunde_Branche'] == 13 else (999 if x['Kunde_Branche'] == 14 else x['Kunde_Geschäftsart']),
    axis=1
)

#fill in Kunde_Sortiment with 999
df['Kunde_Sortiment'] = df['Kunde_Sortiment'].fillna(999)

#fill in Kunde_PLZ with XXX
df['Kunde_PLZ'] = df['Kunde_PLZ'].fillna('XXX')


## Convert to English ##

In [ ]:
# Merge cleaned lookups for names into the main DataFrame
df = df.merge(product_l1[['Titelgruppe', 'Titelgruppe_name']], on='Titelgruppe', how='left')
df = df.merge(product_l2[['Hauptausgabengruppe_aggregiert', 'Hauptausgabengruppe_aggregiert_name']], on='Hauptausgabengruppe_aggregiert', how='left')
df = df.merge(business_type[['Geschäftsart', 'Geschäftsart_name']], on='Geschäftsart', how='left')
df = df.merge(product_l3[['Hauptausgabengruppe_Bezeichnung', 'Hauptausgabengruppe_Bezeichnung_name']], on='Hauptausgabengruppe_Bezeichnung', how='left')
df = df.merge(kunde_branche[['Kunde_Branche', 'Kunde_Branche_name']], on='Kunde_Branche', how='left')
df = df.merge(kunde_geschaeftsart[['Kunde_Geschäftsart', 'Kunde_Geschäftsart_name']], on='Kunde_Geschäftsart', how='left')
df = df.merge(rubrikengruppe[['Rubrikengruppe', 'Rubrikengruppe_name']], on='Rubrikengruppe', how='left')
df = df.merge(kunde_sortiment[['Kunde_Sortiment', 'Kunde_Sortiment_name']], on='Kunde_Sortiment', how='left')
# Set 'Other area' for rows where Kunde_Land is not 28
df.loc[df['Kunde_Land'] != 28, ['Land name', 'Kreis name']] = 'Other area'
#drop rows with  adjusted_sales  NaN
df = df.dropna(subset=['adjusted_sales'])
#for NaN values in Land name and Kreis name, fill with 'Other area'
df['Land name'] = df['Land name'].fillna('Other area')
df['Kreis name'] = df['Kreis name'].fillna('Other area')


In [ ]:
#convert all columns name to English
df.columns = df.columns.str.replace('Jahr', 'Year', regex=False)
df.columns = df.columns.str.replace('Monat', 'Month', regex=False)
df.columns = df.columns.str.replace('Jahr-Monat', 'Year-Month', regex=False)
df.columns = df.columns.str.replace('Titlegruppe_', 'Brand_', regex=False)
df.columns = df.columns.str.replace('Hauptausgabengruppe_aggregiert', 'Product_group', regex=False)
df.columns = df.columns.str.replace('Hauptausgabengruppe_Bezeichnung_', 'Product_', regex=False)
df.columns = df.columns.str.replace('Rubrikengruppe_', 'Ads_group_', regex=False)
df.columns = df.columns.str.replace('Kunde_Branche', 'Customer_industry', regex=False)
df.columns = df.columns.str.replace('Kunde_Geschäftsart', 'Customer_business_type', regex=False)
df.columns = df.columns.str.replace('Kunde_Sortiment', 'Customer_sortiment', regex=False)
df.columns = df.columns.str.replace('Kunde_PLZ', 'Customer_PLZ', regex=False)
df.columns = df.columns.str.replace('Kunde_Land_', 'Customer_country_', regex=False)
df.columns = df.columns.str.replace('Verkauf', 'Sales', regex=False)
df.columns = df.columns.str.replace('Geschäftsart_', 'Business_type_', regex=False)
df.columns = df.columns.str.replace('Geschäftsart', 'Business_type', regex=False)
df.columns = df.columns.str.replace('Rubrikengruppe', 'Ads_group', regex=False)
df.columns = df.columns.str.replace('Kunde_Branche_name', 'Customer_industry_name', regex=False)
df.columns = df.columns.str.replace('Hauptausgabengruppe_Bezeichnung', 'Product', regex=False)



In [ ]:
#add index column based on Year-Month
df['time_index'] = (df['Year-Month'].dt.year - df['Year-Month'].dt.year.min()) * 12 + df['Year-Month'].dt.month
df['time_index'] -= df['time_index'].min() - 1  # start at 1

In [ ]:
df.head(10)

In [ ]:
# save to csv
df.to_csv('Capstone_cleaned.csv', sep=';', index=False)

## Load Data Exogenous Factors, Customer numbers, and Product numbers##

In [ ]:
## Exogenous factors
#ifo_index
ifo_data = """
Year-Month,ifo_index
01-2020,99.6
02-2020,99.0
03-2020,92.9
04-2020,79.3
05-2020,78.6
06-2020,80.9
07-2020,84.3
08-2020,87.7
09-2020,89.5
10-2020,90.5
11-2020,90.3
12-2020,91.8
01-2021,89.6
02-2021,91.1
03-2021,93.3
04-2021,94.2
05-2021,95.7
06-2021,99.8
07-2021,100.6
08-2021,101.7
09-2021,100.9
10-2021,100.6
11-2021,99.5
12-2021,97.3
01-2022,96.7
02-2022,99.0
03-2022,97.2
04-2022,97.4
05-2022,99.6
06-2022,99.4
07-2022,97.9
08-2022,97.7
09-2022,94.9
10-2022,94.5
11-2022,93.4
12-2022,94.3
01-2023,94.5
02-2023,94.2
03-2023,95.5
04-2023,95.1
05-2023,94.8
06-2023,93.7
07-2023,91.3
08-2023,88.9
09-2023,88.8
10-2023,89.2
11-2023,89.4
12-2023,88.6
01-2024,86.9
02-2024,87.0
03-2024,88.1
04-2024,88.9
05-2024,88.3
06-2024,88.2
07-2024,87.1
08-2024,86.4
09-2024,84.4
10-2024,85.6
11-2024,84.2
12-2024,85.0
01-2025,86.0
02-2025,85.0
03-2025,85.7
04-2025,86.4
05-2025,87.5  
06-2025,86.12 
07-2025,86.12 
08-2025,86.35 
09-2025,86.50 
10-2025,86.52 
11-2025,86.52 
12-2025,86.60 
01-2026, 87.5 
02-2026, 87.5 
03-2026, 87.5 
04-2026, 87.5 
05-2026, 87.5
06-2026, 87.5 

"""

# Read into DataFrame
ifo_df = pd.read_csv(StringIO(ifo_data), sep=",")
ifo_df['Year-Month'] = pd.to_datetime(ifo_df['Year-Month'], format="%m-%Y")
display(ifo_df)
#save to csv
ifo_df.to_csv('ifo_index.csv', index=False, sep=';')


In [ ]:
## Saturday and Thursday

# Generate a date range from January 2020 to June 2026
date_range = pd.date_range(start="2020-01-01", end="2026-06-30", freq="D")

# Create a DataFrame from the date range
df = pd.DataFrame(date_range, columns=["Date"])

# Extract Year, Month, and Day of Week
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day_of_Week'] = df['Date'].dt.dayofweek  # Monday=0, Sunday=6
df['Year-Month'] = df['Date'].dt.strftime('%Y-%m')

# Filter to include only Thursdays (3) and Saturdays (5)
df_filtered = df[df['Day_of_Week'].isin([3, 5])]

# Count the number of Thursdays and Saturdays by month
count_by_month = df_filtered.groupby(['Year-Month', 'Day_of_Week']).size().unstack(fill_value=0)

# Rename columns for clarity
count_by_month.columns = ['Thursday', 'Saturday']

# Reset index to get the final table
thu_sat_monthly = count_by_month.reset_index()

display(thu_sat_monthly)
#save to csv
thu_sat_monthly.to_csv('thu_sat_monthly.csv', index=False, sep=';')


In [ ]:
# Holiday date ranges for Summer and Autumn holidays
summer_holiday_range = {
    2020: ('2020-06-29', '2020-08-11'),
    2021: ('2021-07-05', '2021-08-17'),
    2022: ('2022-06-27', '2022-08-09'),
    2023: ('2023-06-22', '2023-08-04'),
    2024: ('2024-07-08', '2024-08-20'),
    2025: ('2025-07-14', '2025-08-26'),
    2026: ('2026-07-20', '2026-09-01'),
}

autumn_holiday_range = {
    2020: ('2020-10-12', '2020-10-24'),
    2021: ('2021-10-11', '2021-10-23'),
    2022: ('2022-10-04', '2022-10-15'),
    2023: ('2023-10-02', '2023-10-14'),
    2024: ('2024-10-14', '2024-10-26'),
    2025: ('2025-10-13', '2025-10-25'),
    2026: ('2026-10-17', '2026-10-31'),
}

# Function to count the number of days within a given holiday period in a specific month
def count_days_in_range(start_date, end_date, year, month):
    # Create a date range for the specified month in the given year
    date_range = pd.date_range(f'{year}-{month:02d}-01', f'{year}-{month:02d}-28', freq='D')
    
    # Filter the date range to only include dates that fall within the holiday range
    holiday_start = pd.to_datetime(start_date)
    holiday_end = pd.to_datetime(end_date)
    
    # Count how many days in the date range fall within the holiday period
    return len(date_range[(date_range >= holiday_start) & (date_range <= holiday_end)])

# Create an empty list to store results
results = []

# Iterate over each year
for year in range(2020, 2027):
    summer_holiday_start, summer_holiday_end = summer_holiday_range[year]
    autumn_holiday_start, autumn_holiday_end = autumn_holiday_range[year]
    
    # Count summer holiday days for June, July, and August
    summer_holiday_june = count_days_in_range(summer_holiday_start, summer_holiday_end, year, 6)
    summer_holiday_july = count_days_in_range(summer_holiday_start, summer_holiday_end, year, 7)
    summer_holiday_august = count_days_in_range(summer_holiday_start, summer_holiday_end, year, 8)
    
    # Count autumn holiday days for October
    autumn_holiday_october = count_days_in_range(autumn_holiday_start, autumn_holiday_end, year, 10)
    
    # For each month, we now calculate the number of holidays for that month
    for month in range(1, 13):
        # Determine which month is being processed and assign the appropriate holiday value
        if month == 6:  # June
            holiday = summer_holiday_june
        elif month == 7:  # July
            holiday = summer_holiday_july
        elif month == 8:  # August
            holiday = summer_holiday_august
        elif month == 10:  # October
            holiday = autumn_holiday_october
        else:
            holiday = 0  # No holiday for other months

        # Append results for this month
        results.append({
            'Year': year,
            'Month': month,
            'holiday': holiday
        })

# Convert the results to a DataFrame
holiday_df = pd.DataFrame(results)

# Create a 'Year-Month' column to use as the index
holiday_df['Year-Month'] = pd.to_datetime(holiday_df['Year'].astype(str) + '-' + holiday_df['Month'].astype(str), format='%Y-%m')

# Drop 'Month' column as it's no longer needed
holiday_df.drop(columns='Month', inplace=True)

# Display the final holiday_df DataFrame
display(holiday_df)

# Save the holiday DataFrame to a CSV file
holiday_df.to_csv('holiday_data.csv', index=False, sep=';')

# EDA # 

In [ ]:
df = pd.read_csv('Capstone_cleaned.csv', sep=';')

In [ ]:
# Define a consistent minimalist graph formatting function for all charts
def set_consistent_plot_style():
    plt.style.use('default')
    plt.rcParams.update({
        'figure.figsize': (8, 5),
        'axes.titlesize': 14,
        'axes.labelsize': 12,
        'axes.titleweight': 'bold',
        'axes.labelweight': 'bold',
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'legend.frameon': False,
        'legend.loc': 'best',
        'axes.edgecolor': 'white',
        'axes.linewidth': 0,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.spines.left': False,
        'axes.spines.bottom': False,
        'axes.grid': False,
        'grid.color': 'white',
        'grid.linestyle': '',
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans']
    })

set_consistent_plot_style()

In [ ]:
## Trend
print_df=df[df['Product_group'] == 1]
digi_df=df[df['Product_group'] == 2]
## Observe the trend of sales overtime 
def plot_time_trends(df, date_col='Year-Month', year_col='Year', quarter_col='Quarter', value_col='adjusted_sales', title_prefix=''):
    
    # --- Monthly Trend ---
    monthly_trend = df.groupby(date_col)[value_col].sum().sort_index()
    x_months = np.arange(len(monthly_trend))
    y_months = monthly_trend.values

    z_months = np.polyfit(x_months, y_months, 1)
    p_months = np.poly1d(z_months)

    plt.figure(figsize=(14, 6))
    plt.plot(monthly_trend.index.astype(str), y_months, label=value_col)
    plt.plot(monthly_trend.index.astype(str), p_months(x_months), 'r--', label='Trendline')
    plt.title(f"{title_prefix} Monthly Trend")
    plt.xlabel("Month-Year")
    plt.ylabel("Sales (Sum)")
    plt.legend()
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# For all data
plot_time_trends(df, title_prefix="Total")
# For digital only
plot_time_trends(digi_df, title_prefix="Digital")
# For printed only
plot_time_trends(print_df, title_prefix="Print")

In [ ]:
#plot sales trend by area
def plot_sales_by_area(df, area_col='area', date_col='Year-Month', value_col='adjusted_sales'):

    # Group by area and date, then sum the sales
    area_trend = df.groupby([area_col, date_col])[value_col].sum().unstack(level=0)

    # Plot each area's trend
    plt.figure(figsize=(14, 8))
    for area in area_trend.columns:
        plt.plot(area_trend.index.astype(str), area_trend[area], label=area)

    plt.title("Sales Trend by Area")
    plt.xlabel("Month-Year")
    plt.ylabel("Sales (Sum)")
    plt.legend()
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Plot sales trend by area
plot_sales_by_area(df)
plot_sales_by_area(digi_df)
plot_sales_by_area(print_df)

In [ ]:
#Seasonality decomposition

def seasonal_decompose_adjusted_sales(df, year_col='Year', month_col='Month_num', value_col='adjusted_sales'):
    df_small = df[[year_col, month_col, value_col]].copy()

    # Convert types
    df_small[year_col] = df_small[year_col].astype(int)
    df_small[month_col] = df_small[month_col].astype(int)

    # Create datetime
    df_small['Year-Month'] = pd.to_datetime(
        df_small[year_col].astype(str) + '-' + df_small[month_col].astype(str),
        format='%Y-%m'
    )

    # Group to monthly time series
    monthly_sales = df_small.groupby('Year-Month')[value_col].sum().sort_index()

    # Set frequency and interpolate
    monthly_sales = monthly_sales.asfreq('MS').interpolate()

    # Decompose
    result = seasonal_decompose(monthly_sales, model='additive', period=12)
    result.plot()

     # Try to get the variable name of the DataFrame passed to the function
    frame = inspect.currentframe()
    try:
        arg_name = None
        for k, v in frame.f_back.f_locals.items():
            if id(v) == id(df):
                arg_name = k
                break
        if arg_name is not None:
            plt.suptitle(f'Seasonal Decomposition of {arg_name}', fontsize=16)
        else:
            plt.suptitle('Seasonal Decomposition', fontsize=16)
    finally:
        del frame
    plt.tight_layout()
    plt.show()

seasonal_decompose_adjusted_sales(df)

In [ ]:
#Distribution by sector

# Group data
grouped = df.groupby(
    ['Customer_industry_name', 'Product_group_name']
)['adjusted_sales'].sum().reset_index()

# Pivot for stacking
pivot = grouped.pivot(
    index='Customer_industry_name',
    columns='Product_group_name',
    values='adjusted_sales'
).fillna(0)

# sort sectors by total adjusted sales
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=True).index]

# Plot stacked horizontal bar
colors = ['#3b6fb6', "#97b7e0"]  
ax = pivot.plot(
    kind='barh',
    stacked=True,
    figsize=(12, 8),
    color=colors
)

plt.title("Adjusted Sales by Sector and Product Type")
plt.xlabel("Total Adjusted Sales")
plt.ylabel("Customer Sector")
plt.legend(title="Product Type")
plt.tight_layout()
plt.show()


# Macro Forecast - SARIMAX #

In [ ]:
#import & process external data
df = pd.read_csv('Capstone_cleaned.csv', sep=';')
ifo_trade = pd.read_csv('ifo_trade.csv', sep=',')
ifo_service = pd.read_csv('ifo_service.csv', sep=',')
ifo_index = pd.read_csv('ifo_index.csv', sep=';')


# Rename columns if necessary
ifo_index.columns = [col.strip() for col in ifo_index.columns]
ifo_trade.columns = [col.strip() for col in ifo_trade.columns]
ifo_service.columns = [col.strip() for col in ifo_service.columns]
ifo_trade = ifo_trade.rename(columns={'adjusted_sales': 'ifo_trade'})
ifo_service = ifo_service.rename(columns={'adjusted_sales': 'ifo_service'})


# Convert 'Year-Month' to datetime
ifo_index.rename(columns={'Jahr-Monat': 'Year-Month'}, inplace=True)
ifo_index['Year-Month'] = pd.to_datetime(ifo_index['Year-Month'])
ifo_trade['Year-Month'] = pd.to_datetime(ifo_trade['Year-Month'])
ifo_service['Year-Month'] = pd.to_datetime(ifo_service['Year-Month'])

#school holiday
holiday_data = pd.read_csv('holiday_data.csv', sep=';')



In [ ]:
#Create monthly_df by grouping by Year-Month and summing adjusted_sales, adjusted_sales
monthly_df = df.groupby('Year-Month').agg({
    'adjusted_sales': 'sum',
}).reset_index()
# Ensure 'Year-Month' is datetime before extracting month
monthly_df['Year-Month'] = pd.to_datetime(monthly_df['Year-Month'])
#create month_num column
monthly_df['month_num'] = monthly_df['Year-Month'].dt.month

In [ ]:
# Ensure 'Year-Month' column exists in all DataFrames
#monthly_df['Year-Month'] = pd.to_datetime(monthly_df['Date']).dt.to_period('M').dt.to_timestamp()
#ifo_index['Year-Month'] = pd.to_datetime(ifo_index['Date']).dt.to_period('M').dt.to_timestamp()
#holiday_data['Year-Month'] = pd.to_datetime(holiday_data['Date']).dt.to_period('M').dt.to_timestamp()

# Convert to datetime if not already
monthly_df['Year-Month'] = pd.to_datetime(monthly_df['Year-Month'])
ifo_index['Year-Month'] = pd.to_datetime(ifo_index['Year-Month'])
holiday_data['Year-Month'] = pd.to_datetime(holiday_data['Year-Month'])

# Merge DataFrames on 'Year-Month'
monthly_df = pd.merge(monthly_df, ifo_index, on='Year-Month', how='left')
monthly_df = pd.merge(monthly_df, holiday_data, on='Year-Month', how='left')


## Diagnosis ##

In [ ]:
#check outliers of adjusted_sales of monthly_df
monthly_df['adjusted_sales'].describe()
# Create a boxplot to visualize outliers
plt.figure(figsize=(10, 6))
sns.boxplot(x=monthly_df['adjusted_sales'])
plt.title('Boxplot of Adjusted Sales')
plt.xlabel('Adjusted Sales')
plt.show()

In [ ]:
#plot trend of monthly_df
plt.figure(figsize=(12, 6))
plt.plot(monthly_df['Year-Month'], monthly_df['adjusted_sales'], marker='o', linestyle='-')
plt.title('Trend of Adjusted Sales Over Time')
plt.xlabel('Year-Month')
plt.ylabel('Adjusted Sales')
plt.xticks(rotation=45)
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Create exogenous variables
#create other exogenous variables
monthly_df['lockdown_2020'] = (monthly_df['Year-Month'] == pd.to_datetime('2021-01-01')).astype(int)  # Lockdown Dec 2020 - Jan 2021
monthly_df['recover_flood_2021'] = ((monthly_df['Year-Month'] >= pd.to_datetime('2021-08-01')) &
                                    (monthly_df['Year-Month'] <= pd.to_datetime('2021-11-30'))).astype(int)
monthly_df['restructuring_2023'] = (monthly_df['Year-Month'] >= pd.to_datetime('2023-06-01')).astype(int)  # Internal restructuring in June-July 2023
monthly_df['holiday'] = monthly_df['holiday'].fillna(0)  # Fill NaN values in holiday_effect with 0

In [ ]:
#drop rows with NaN if any
monthly_df = monthly_df.dropna(subset=['adjusted_sales']).copy()

In [ ]:
#check outliers by month
plt.figure(figsize=(12, 6))
sns.boxplot(x='month_num', y='adjusted_sales', data=monthly_df)
plt.title('Boxplot of Adjusted Sales by Month')
plt.xlabel('Month')
plt.ylabel('Adjusted Sales')
plt.xticks(ticks=np.arange(1, 13), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.show()

In [ ]:
#check corrrelation of exogenous variables
corr_matrix = monthly_df.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Exogenous Variables')
plt.show()

In [ ]:
## Check differencing for stationarity

#Prepare series 
series = monthly_df['adjusted_sales'].dropna()

# ADF Test: Original series 
adf_result = adfuller(series)
print("ADF Test (Original Series)")
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")
print("=> Non-stationary" if adf_result[1] > 0.05 else "=> Stationary")
print("-" * 50)

# ADF Test: First Difference 
first_diff = series.diff().dropna()
adf_diff = adfuller(first_diff)
print("ADF Test (First Difference)")
print(f"ADF Statistic: {adf_diff[0]:.4f}")
print(f"p-value: {adf_diff[1]:.4f}")
print("=> Non-stationary" if adf_diff[1] > 0.05 else "=> Stationary")
print("-" * 50)

# ADF Test: Seasonal Difference (lag 12) 
seasonal_diff = series.diff(12).dropna()
adf_seasonal = adfuller(seasonal_diff)
print("ADF Test (Seasonal Difference: 12 lags)")
print(f"ADF Statistic: {adf_seasonal[0]:.4f}")
print(f"p-value: {adf_seasonal[1]:.4f}")
print("=> Non-stationary" if adf_seasonal[1] > 0.05 else "=> Stationary")
print("-" * 50)

# Plot ACF & PACF for First Differenced Series 
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
plot_acf(first_diff, lags=24, ax=axes[0])
axes[0].set_title("ACF of First Difference (to justify q)")
plot_pacf(first_diff, lags=24, ax=axes[1])
axes[1].set_title("PACF of First Difference (to justify p)")
plt.tight_layout()
plt.show()

# Plot ACF & PACF for Seasonal Differenced Series 
max_lags = min(24, len(seasonal_diff) // 2 - 1)  # Ensure lags < 50% of sample size
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
plot_acf(seasonal_diff, lags=max_lags, ax=axes[0])
axes[0].set_title("ACF of Seasonal Difference (to justify Q)")
plot_pacf(seasonal_diff, lags=max_lags, ax=axes[1])
axes[1].set_title("PACF of Seasonal Difference (to justify P)")
plt.tight_layout()
plt.show()

## Baseline Model ##

In [ ]:
# Features (use a simple index as placeholder, or use existing exogenous features)
X = monthly_df.drop(columns=['adjusted_sales'])  # if you have exogenous features
y = monthly_df['adjusted_sales']

dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X, y)
y_pred = dummy_model.predict(X)

# Metrics
baseline_r2 = r2_score(y, y_pred)
baseline_mae = mean_absolute_error(y, y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"Baseline Regression Model - R2: {baseline_r2:.4f}, MAE: {baseline_mae:.2f}, RMSE: {baseline_rmse:.2f}")

## Model SARIMAX ##

In [ ]:
# Define target and exogenous variables 
exog_variables= ['ifo_index', 
    'lockdown_2020',
    'holiday',
    'recover_flood_2021',
    'restructuring_2023',
]
y = monthly_df['adjusted_sales']
X = monthly_df[exog_variables]

In [ ]:
# Ensure 'Year-Month' is a column and in datetime format
if 'Year-Month' not in monthly_df.columns:
    monthly_df = monthly_df.reset_index()

monthly_df['Year-Month'] = pd.to_datetime(monthly_df['Year-Month'])

# Define the split point for the last 12 months
split_test = monthly_df['Year-Month'].max() - pd.DateOffset(months=13)

# Create masks based on the 'Year-Month' column
train_mask = monthly_df['Year-Month'] < split_test
test_mask = monthly_df['Year-Month'] >= split_test

# Reset index of y and X to align correctly with the DataFrame
y = y.reset_index(drop=True)
X = X.reset_index(drop=True)

# Ensure masks are applied correctly by using .loc on their index positions
y_train = y.loc[train_mask.values]
X_train = X.loc[train_mask.values]
y_test = y.loc[test_mask.values]
X_test = X.loc[test_mask.values]


In [ ]:
start_date = '2020-01-01'
periods = len(y_train)  # 

# Create a datetime index with monthly frequency (MS: Month Start)
datetime_index = pd.date_range(start=start_date, periods=periods, freq='MS')

# Step 2: Assign the new datetime index to both y_train and X_train
y_train.index = datetime_index
X_train.index = datetime_index

# Check the new index
print(f"New y_train index: {y_train.index[:5]}")
print(f"New X_train index: {X_train.index[:5]}")

# Start date for the test set (the month after the last date of the training set)
start_date_test = datetime_index[-1] + pd.DateOffset(months=1)
periods_test = len(y_test)  # The number of periods in the test set

# Create the datetime index for the test set (monthly frequency, MS: Month Start)
datetime_index_test = pd.date_range(start=start_date_test, periods=periods_test, freq='MS')

# Assign the new datetime index to y_test and X_test
y_test.index = datetime_index_test
X_test.index = datetime_index_test

print(f"New y_test index (first 5): {y_test.index[:5]}")
print(f"New X_test index (first 5): {X_test.index[:5]}")

In [ ]:
# --- Parameters for walk-forward cross-validation ---
initial_train_size = 48  # Number of initial months in training (e.g., 4 years)
n_splits = 4             # Number of walk-forward iterations
forecast_horizon = 1     # Forecast 1 step ahead in each iteration

# --- Define SARIMAX parameter grid ---
p = d = q = range(0, 2)
P = D = Q = range(0, 2)
seasonal_period = 12

pdq = list(itertools.product(p, d, q))
seasonal_pdq = list(itertools.product(P, D, Q))

best_avg_aic = float('inf')
best_order = None
best_seasonal_order = None
best_model = None

# --- Grid Search with Walk-Forward CV ---
for order in pdq:
    for seasonal_order in seasonal_pdq:
        aic_scores = []

        try:
            for split in range(n_splits):
                # Define rolling train/test split
                train_end = initial_train_size + split
                y_train_split = y_train.iloc[:train_end]
                y_test_split = y_train.iloc[train_end:train_end + forecast_horizon]
                X_train_split = X_train.iloc[:train_end]
                X_test_split = X_train.iloc[train_end:train_end + forecast_horizon]

                # Ensure matching shapes and proper index alignment
                model = sm.tsa.SARIMAX(
                    y_train_split,
                    exog=X_train_split,
                    order=order,
                    seasonal_order=(seasonal_order[0], seasonal_order[1], seasonal_order[2], seasonal_period),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                results = model.fit(disp=False)
                aic_scores.append(results.aic)

            # Compute average AIC across folds
            avg_aic = np.mean(aic_scores)

            # Update best model if lower AIC is found
            if avg_aic < best_avg_aic:
                best_avg_aic = avg_aic
                best_order = order
                best_seasonal_order = seasonal_order
                best_model = results  # Keep last fitted model from final split

        except Exception as e:
            continue  # Silently skip invalid configurations

# --- Final Output ---
print(f"\nBest SARIMAX order: {best_order}, seasonal_order: {best_seasonal_order}, Avg AIC: {best_avg_aic:.2f}")

if best_model is not None:
    print(best_model.summary())
else:
    print("No valid SARIMAX model was found.")


In [ ]:
##Evaluation 
 
# --- 1. Get the forecasted values for the test period ---
forecast = best_model.forecast(steps=len(y_test), exog=X_test)

# --- 2. Calculate the evaluation metrics ---
# R-squared (Coefficient of Determination)
r2 = r2_score(y_test, forecast)

# Mean Squared Error (MSE)
mse = mean_squared_error(y_test, forecast)

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, forecast)

# AIC (from the model summary)
aic = best_model.aic

# --- 3. Print the evaluation results ---
print("Model Evaluation:")
print(f"  R²: {r2:.4f}")
print(f"  AIC: {aic:.2f}")
print(f"  MSE: {mse:.2f}")
print(f"  MAE: {mae:.2f}")


In [ ]:
# --- 4. Plot the actual vs forecasted values ---
plt.figure(figsize=(14, 6))

# Plot the actual adjusted sales (train + validation data)
plt.plot(y_train.index, y_train, label='Historical Adjusted Sales (Train+Val)', color='blue')

# Plot the actual test data (test set)
plt.plot(y_test.index, y_test, label='Actual Adjusted Sales (Test)', color='green')

# Plot the forecasted values
plt.plot(y_test.index, forecast, label='Forecasted Adjusted Sales', color='orange', linestyle='--')

# Add a vertical line to indicate where the forecast starts
plt.axvline(x=y_test.index.min(), color='gray', linestyle='--', label='Forecast Start')

# Add title and labels
plt.title('SARIMAX Forecast vs Actual Adjusted Sales')
plt.xlabel('Date')
plt.ylabel('Adjusted Sales')

# Display legend
plt.legend()

# Adjust layout and rotate x-axis labels for better readability
plt.tight_layout()
plt.xticks(rotation=45)

# Show the plot
plt.show()

In [ ]:
#Forecasting 

#Create the future datetime index for the next 18 months (starting from 2025-01-01)
start_date_future = pd.to_datetime('2025-05-01')
periods_future = 18  # Forecast for the next 18 months

# Create the future datetime index for 18 months
datetime_index_future = pd.date_range(start=start_date_future, periods=periods_future, freq='MS')

# Prepare the exogenous variables (X) for the next 18 months.
# Use the last available exogenous values (for simplicity)
last_exog = X_test.iloc[-1:].copy()

# Repeat the last observed exogenous variable values for the next 18 months
X_future = last_exog.loc[last_exog.index.repeat(periods_future)].reset_index(drop=True)
if 'scaler' in globals():
    X_future_scaled = pd.DataFrame(
        scaler.transform(X_future),
        columns=X_test.columns
    )
else:
    X_future_scaled = X_future

# Forecast using the trained SARIMAX model
forecast_sarimax = best_model.forecast(steps=periods_future, exog=X_future_scaled)

# Plot the actual vs forecasted values
plt.figure(figsize=(12, 6))

# Plot historical adjusted sales (train + test data)
plt.plot(monthly_df['Year-Month'], monthly_df['adjusted_sales'], label='Historical Adjusted Sales', color='blue')

# Plot the forecasted values (for the next 18 months)
plt.plot(datetime_index_future, forecast_sarimax, label='Forecasted Adjusted Sales', color='orange', linestyle='--')

# Add labels and title
plt.axvline(x=datetime_index_future[0], color='gray', linestyle='--', label='Forecast Start')
plt.title('SARIMAX Forecast for the Next 18 Months')
plt.xlabel('Date')
plt.ylabel('Adjusted Sales')
plt.xlim([pd.to_datetime('2020-01-01'), datetime_index_future[-1]])
plt.legend()

# Show the plot
plt.plot(monthly_df['Year-Month'], monthly_df['adjusted_sales'], label='Historical Adjusted Sales', color='blue')
plt.tight_layout()
plt.xticks(rotation=45)
plt.savefig('overall_all.png', dpi=300)
plt.show()


# Clustering #

In [ ]:
df.info()

## Preparation and Filtering ##

### Filtering and Aggregation ###

In [ ]:
### Filtering for relevant Customer Characteristics ###
df_filter = df[
    df['area'].isin(['Cologne', 'East', 'West']) & # Filter Area (Cologne, West, East)
    df['Ads_group'].isin([1.0, 3.0, 4.0, 6.0, 8.0, 17.0]) & # Filter Ads_Group (Recommendation Ads,Obituary Market,Job Market,Real Estate Market,Public Notices, Commercial Register,Events)
    df['Customer_industry'].isin([1.0, 2.0, 3.0, 4.0, 6.0, 10.0, 12.0]) # Filter Industry (Services, Trade, Internal, Real Estate, Public Institutions/Associations, Tourism/Leisure, Health)
]


### Aggregation ###

# Group and aggregate adjusted sales and transaction count
df_agg_monthly_sales = (
    df_filter.groupby(['Year-Month', 'Product_group', 'Customer_industry', 'area', 'Ads_group'])
    .agg(
        Avg_Adjusted_Sales=('adjusted_sales', 'mean'),
        Sum_Adjusted_Sales=('adjusted_sales', 'sum'),
        Transactions=('adjusted_sales', 'count'),
    )
    .reset_index()
)

# Merge industry counts
df_agg_monthly_sales = df_agg_monthly_sales.merge(
    monthly_customer_industry,
    on=['Year-Month', 'Customer_industry'],
    how='left'
)

# Compute adjusted sales per transaction
df_agg_monthly_sales['Adjusted_Sales_per_Customer'] = (
    df_agg_monthly_sales['Sum_Adjusted_Sales'] / df_agg_monthly_sales['Value']
)

df_agg_monthly_sales = df_agg_monthly_sales.rename(columns= {'Value': 'customer number Industry'})
df_agg_monthly_sales.head()

### Normalized Feature Matrix ###

In [ ]:
### Feature Matrix ###

df_cluster_base = df_agg_monthly_sales.pivot_table(
    index=['Product_group', 'Customer_industry', 'area'], # Included Features
    columns='Year-Month', # monthly level
    values='Avg_Adjusted_Sales', # Metric
    fill_value=0
)

#df_cluster_base.head(30)

### Normalize Feature Matrix ###
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster_base)

## Hierachical Clustering ##

In [ ]:
k_range = range(2, 11)
silhouette_scores = []
wcss = []

for k in k_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(X_scaled)

    # Silhouette Score
    sil_score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil_score)

    # WCSS: Approximate with sum of squared distances to cluster means
    cluster_centers = []
    for cluster_id in np.unique(labels):
        cluster_points = X_scaled[labels == cluster_id]
        center = cluster_points.mean(axis=0)
        cluster_centers.append(center)
    
    wcss_k = np.sum([
        np.sum((X_scaled[labels == i] - center) ** 2)
        for i, center in zip(np.unique(labels), cluster_centers)
    ])
    wcss.append(wcss_k)

# Plotting
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Plot WCSS
ax[0].plot(k_range, wcss, marker='o')
ax[0].set_title('WCSS vs. Number of Clusters (Hierarchical)')
ax[0].set_xlabel('Number of Clusters (k)')
ax[0].set_ylabel('Within-Cluster Sum of Squares (WCSS)')
ax[0].grid(True)

# Plot Silhouette Scores
ax[1].plot(k_range, silhouette_scores, marker='o', color='orange')
ax[1].set_title('Silhouette Score vs. Number of Clusters (Hierarchical)')
ax[1].set_xlabel('Number of Clusters (k)')
ax[1].set_ylabel('Silhouette Score')
ax[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
### Comparison different Linkage Methods ###

linkage_methods = ['ward', 'complete', 'average', 'single']
results = []

for method in linkage_methods:
    # For 'ward', skip metric to avoid conflict
    if method == 'ward':
        model = AgglomerativeClustering(n_clusters=3, linkage=method)
    else:
        model = AgglomerativeClustering(n_clusters=3, linkage=method, metric='euclidean')

    # Fit model
    labels = model.fit_predict(X_scaled)

    # Calculate silhouette score
    sil_score = silhouette_score(X_scaled, labels)

    # Estimate WCSS (only valid approximation for 'ward' or centroid-based models)
    cluster_centers = []
    for i in np.unique(labels):
        cluster_points = X_scaled[labels == i]
        cluster_center = cluster_points.mean(axis=0)
        cluster_centers.append(cluster_center)
    wcss = np.sum([
        np.sum((X_scaled[labels == i] - center) ** 2)
        for i, center in zip(np.unique(labels), cluster_centers)
    ])

    # Store results
    results.append({
        'linkage': method,
        'silhouette_score': sil_score,
        'wcss': wcss
    })

# Display comparison
import pandas as pd
df_linkage_comparison = pd.DataFrame(results)
print(df_linkage_comparison.sort_values(by='wcss', ascending=True))


In [ ]:
# Fit Agglomerative (Hierarchical) Clustering model with Ward linkage
agg_model = AgglomerativeClustering(
    n_clusters=3,                # Set the desired number of clusters
    linkage='ward'               # Ward linkage implicitly uses Euclidean distance
)

# Predict cluster labels
cluster_labels_hierarchical = agg_model.fit_predict(X_scaled)

# Create result DataFrame with hierarchical cluster labels
df_hierarchical_results = df_cluster_base.copy()
df_hierarchical_results['cluster_hierarchical'] = cluster_labels_hierarchical

# Show number of points per cluster
df_hierarchical_results['cluster_hierarchical'].value_counts()


## K-means Clustering ##

### Selection of k with Elbow Method ###

In [ ]:
inertia = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X_scaled)
    
    inertia.append(model.inertia_)  # WCSS
    sil_score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil_score)

# Plot WCSS and Silhouette side-by-side
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Elbow plot (WCSS)
ax[0].plot(k_range, inertia, marker='o')
ax[0].set_title('Elbow Method (WCSS)')
ax[0].set_xlabel('Number of Clusters (k)')
ax[0].set_ylabel('WCSS (Inertia)')
ax[0].grid(True)

# Silhouette Score plot
ax[1].plot(k_range, silhouette_scores, marker='o', color='orange')
ax[1].set_title('Silhouette Score vs. k')
ax[1].set_xlabel('Number of Clusters (k)')
ax[1].set_ylabel('Silhouette Score')
ax[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Apply K-Means with selected number of clusters
kmeans_model = KMeans(n_clusters=3, random_state=42)
cluster_labels_kmeans = kmeans_model.fit_predict(X_scaled)

# Attach K-Means cluster labels to original data
df_kmeans_results = df_cluster_base.copy()
df_kmeans_results['cluster_kmeans'] = cluster_labels_kmeans

# Reset index to access original identifiers
df_kmeans_results = df_kmeans_results.reset_index()

In [ ]:
df_kmeans_results['cluster_kmeans'].value_counts()

# Model Evaluation and Comparison #

In [ ]:
### Compare both Models with Silhouette Score and Within-Cluster Sum of Squares for k=3 ###

# --- K-Means for k=3 ---
kmeans_model = KMeans(n_clusters=3, random_state=42)
labels_kmeans = kmeans_model.fit_predict(X_scaled)
sil_kmeans = silhouette_score(X_scaled, labels_kmeans)
wcss_kmeans = kmeans_model.inertia_  # Directly available from KMeans

# --- Hierarchical Clustering (Ward) for k=3 ---
hier_model = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_hier = hier_model.fit_predict(X_scaled)
sil_hier = silhouette_score(X_scaled, labels_hier)

# WCSS approximation for hierarchical
cluster_centers_hier = []
for i in np.unique(labels_hier):
    cluster_points = X_scaled[labels_hier == i]
    center = cluster_points.mean(axis=0)
    cluster_centers_hier.append(center)

wcss_hier = np.sum([
    np.sum((X_scaled[labels_hier == i] - center) ** 2)
    for i, center in zip(np.unique(labels_hier), cluster_centers_hier)
])

# --- Create Comparison Table ---
df_clustering_comparison = pd.DataFrame([
    {
        'method': 'KMeans',
        'n_clusters': 3,
        'silhouette_score': sil_kmeans,
        'wcss': wcss_kmeans
    },
    {
        'method': 'Hierarchical (Ward)',
        'n_clusters': 3,
        'silhouette_score': sil_hier,
        'wcss': wcss_hier
    }
])

print(df_clustering_comparison)


# Interpretation of k-means Clustering Results #

In [ ]:
df_kmeans_results['cluster_kmeans'].value_counts()

In [ ]:
### Preparation and Merge for Plotting ###

# Ensure df_kmeans_results has the correct index or reset it if needed
df_kmeans_results = df_kmeans_results.reset_index()

# Merge K-Means cluster labels with the original sales data
df_clustered = df_agg_monthly_sales.merge(
    df_kmeans_results[['Product_group', 'Customer_industry', 'area', 'cluster_kmeans']],
    on=['Product_group', 'Customer_industry', 'area'],
    how='left'
)

### Plot Analysis: Volume, Value and Number of Customer ###

In [ ]:
# --- Group and compute means ---
# 1. Avg Adjusted Sales
df_cluster_monthly_sales = df_clustered.groupby(['cluster_kmeans', 'Year-Month'])['Sum_Adjusted_Sales'].mean().reset_index()

# 2. Avg Transactions
df_cluster_monthly_trans = df_clustered.groupby(['cluster_kmeans', 'Year-Month'])['Transactions'].mean().reset_index()

# 3. Avg Customers
df_cluster_monthly_customers = df_clustered.groupby(['cluster_kmeans', 'Year-Month'])['customer number Industry'].mean().reset_index()

# --- Set up subplots ---
fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)

# Plot 1: Avg Sum_Adjusted_Sales
for cluster_id in sorted(df_cluster_monthly_sales['cluster_kmeans'].unique()):
    cluster_data = df_cluster_monthly_sales[df_cluster_monthly_sales['cluster_kmeans'] == cluster_id]
    axes[0].plot(cluster_data['Year-Month'], cluster_data['Sum_Adjusted_Sales'], label=f'Cluster {cluster_id}', marker='o')

axes[0].set_title('Avg. Sum_Adjusted_Sales Over Time by K-Means Cluster')
axes[0].set_ylabel('Avg. Adjusted Sales')
axes[0].grid(True)
axes[0].legend(title='Cluster')

# Plot 2: Avg Transactions
for cluster_id in sorted(df_cluster_monthly_trans['cluster_kmeans'].unique()):
    cluster_data = df_cluster_monthly_trans[df_cluster_monthly_trans['cluster_kmeans'] == cluster_id]
    axes[1].plot(cluster_data['Year-Month'], cluster_data['Transactions'], label=f'Cluster {cluster_id}', marker='o')

axes[1].set_title('Avg. Number of Transactions Over Time by K-Means Cluster')
axes[1].set_ylabel('Avg. Transactions')
axes[1].grid(True)

# Plot 3: Avg Customer Number
for cluster_id in sorted(df_cluster_monthly_customers['cluster_kmeans'].unique()):
    cluster_data = df_cluster_monthly_customers[df_cluster_monthly_customers['cluster_kmeans'] == cluster_id]
    axes[2].plot(cluster_data['Year-Month'], cluster_data['customer number Industry'], label=f'Cluster {cluster_id}', marker='o')

axes[2].set_title('Avg. Customer Number per Industry Over Time by K-Means Cluster')
axes[2].set_xlabel('Year-Month')
axes[2].set_ylabel('Avg. Customers')
axes[2].grid(True)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


### Composition Analysis: Customer Industry and Area percentage in each Cluster ###

In [ ]:

# Count number of entries per (cluster, industry)
df_industry_counts = df_clustered.groupby(['cluster_kmeans', 'Customer_industry_name']).size().reset_index(name='count')

# Calculate percentage within each cluster
df_industry_counts['percentage'] = df_industry_counts.groupby('cluster_kmeans')['count'].transform(lambda x: 100 * x / x.sum())

# Pivot for plotting (clusters as columns, industries as rows)
df_industry_pct = df_industry_counts.pivot(index='Customer_industry_name', columns='cluster_kmeans', values='percentage').fillna(0)

# Plot bar chart for each cluster (grouped bars by industry)
df_industry_pct.plot(kind='bar', figsize=(12, 6))
plt.title('Industry Composition by K-Means Cluster (%)')
plt.xlabel('Customer Industry')
plt.ylabel('Percentage of Cluster')
plt.xticks(rotation=45)
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Count number of entries per (cluster, area)
df_area_counts = df_clustered.groupby(['cluster_kmeans', 'area']).size().reset_index(name='count')

# Calculate percentage within each cluster
df_area_counts['percentage'] = df_area_counts.groupby('cluster_kmeans')['count'].transform(lambda x: 100 * x / x.sum())

# Pivot for plotting (clusters as columns, areas as rows)
df_area_pct = df_area_counts.pivot(index='area', columns='cluster_kmeans', values='percentage').fillna(0)

# Plot bar chart for each cluster (grouped bars by area)
df_area_pct.plot(kind='bar', figsize=(12, 6))
plt.title('Area Composition by K-Means Cluster (%)')
plt.xlabel('Area')
plt.ylabel('Percentage of Cluster')
plt.xticks(rotation=45)
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### Interpretation and Naming of Clusters: ###

Cluster 0: “High Value & High Volume - High engagement Customers across several Industries”

Cluster 1: “Low Value & High Volume - Small Business Customers”

Cluster 2: “High Value & Customer Volume – Service & Trade Customers”

In [ ]:

# Define custom names for each cluster
cluster_names = {
    0: "High Value & High Volume - High engagement Customers across several Industries",
    1: "Low Value & High Volume - Small Business Customers",
    2: "High Value & Customer Volume – Service & Trade Customers"
}

# Create the summary table
df_cluster_summary = df_clustered.groupby('cluster_kmeans').agg({
    'Customer_industry_name': lambda x: ', '.join(sorted(x.dropna().astype(str).unique())),
    'Sum_Adjusted_Sales': 'mean',
    'Transactions': 'mean',
    'customer number Industry': 'mean'
}).reset_index()


# Calculate additional metrics
df_cluster_summary['Avg_Sales_per_Transaction'] = (
    df_cluster_summary['Sum_Adjusted_Sales'] / df_cluster_summary['Transactions']
)

# Add cluster names
df_cluster_summary['Cluster_Description'] = df_cluster_summary['cluster_kmeans'].map(cluster_names)

# Rename and reorder columns
df_cluster_summary = df_cluster_summary.rename(columns={
    'cluster_kmeans': 'Cluster',
    'Customer_industry_name': 'Industries',
    'Sum_Adjusted_Sales': 'Avg_Adjusted_Sales (€)',
    'Transactions': 'Avg_Transactions',
    'customer number Industry': 'Avg_Customer_Number'
})[[
    'Cluster', 'Cluster_Description', 'Industries',
    'Avg_Adjusted_Sales (€)', 'Avg_Transactions',
    'Avg_Sales_per_Transaction', 'Avg_Customer_Number'
]]

# Format for display
pd.set_option('display.float_format', '{:,.2f}'.format)
print(df_cluster_summary.to_string(index=False))



In [ ]:
# Initialize the Dash app
app = JupyterDash(__name__)

# Prepare the data
df_display = df_cluster_summary.copy()
df_display['Cluster'] = df_display['Cluster'].astype(str)

# Define app layout
app.layout = html.Div([
    html.H2("K-Means Cluster Summary", style={'textAlign': 'center'}),
    dash_table.DataTable(
        columns=[
            {"name": col, "id": col, "type": "text" if df_display[col].dtype == "object" else "numeric", "format": {'specifier': ',.2f'} if '€' in col or 'Avg' in col else None}
            for col in df_display.columns
        ],
        data=df_display.to_dict('records'),
        style_cell={'padding': '10px', 'textAlign': 'left', 'fontFamily': 'Arial'},
        style_header={'backgroundColor': '#f2f2f2', 'fontWeight': 'bold'},
        style_table={'overflowX': 'auto'},
        style_data_conditional=[
            {
                'if': {'row_index': 'odd'},
                'backgroundColor': '#fafafa'
            }
        ],
        page_size=10
    )
])

# Run the app inline in Jupyter
app.run(mode='inline', port=8061)



# Industry Deep Dive by area - XGboost #

In [ ]:
#create dataset for Services and Trade industries
services_df = df[df['Customer_industry_name'] == 'Services']
trade_df = df[df['Customer_industry_name'] == 'Trade']
#seasonal decomposition for Services and Trade industries
seasonal_decompose_adjusted_sales(services_df)
seasonal_decompose_adjusted_sales(trade_df)

In [ ]:
#add area_ to df
df['area_num'] = df['area'].map({
    'Cologne': 1, 'West': 2, 'East':3, 'National/International': 4
})

In [ ]:
#Create separate dataframes
df['Year-Month'] = pd.to_datetime(df['Year-Month'], format='%Y-%m')
df_trade = df[df['Customer_industry_name'] == 'Trade'].copy()
df_trade = df_trade.groupby(['area_num','Year-Month']).agg({
    'adjusted_sales': 'sum'
}).reset_index()

df_service = df[df['Customer_industry_name'] == 'Services'].copy()
df_service = df_service.groupby(['area_num','Year-Month']).agg({
    'adjusted_sales': 'sum'
}).reset_index()

## Diagnosis ##

In [ ]:
# Function to detect outliers and add 'outliers' column to the DataFrame
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Create a new column 'outliers' where True indicates an outlier
    df['outliers'] = (df[column] < lower_bound) | (df[column] > upper_bound)
    return df

# Add 'outliers' column to each DataFrame
df_trade = detect_outliers(df_trade, 'adjusted_sales')
df_service = detect_outliers(df_service, 'adjusted_sales')

# Plotting the data with outliers marked as red dots
def plot_data(df, title):
    plt.figure(figsize=(12, 6))
    
    # Plot adjusted sales data
    sns.lineplot(data=df, x='Year-Month', y='adjusted_sales', label='Adjusted Sales')
    
    plt.title(title)
    plt.xlabel('Year-Month')
    
    # Mark outliers if present
    outlier_points = df[df['outliers'] == True]  # Filter rows where 'outliers' is True
    if not outlier_points.empty:
        plt.scatter(outlier_points['Year-Month'], outlier_points['adjusted_sales'], color='red', label='Outliers', zorder=5)

    plt.ylabel('Adjusted Sales')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Plotting the data for Trade and Services
plot_data(df_trade, 'Trade')
plot_data(df_service, 'Service')

## Function Define ##

In [ ]:
##Add exogenous factors
def add_exog_factor(df: pd.DataFrame, ifo_data: pd.DataFrame, holiday_data: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Year-Month'] = pd.to_datetime(df['Year-Month'])
    ifo_data['Year-Month'] = pd.to_datetime(ifo_data['Year-Month'])
    holiday_data['Year-Month'] = pd.to_datetime(holiday_data['Year-Month'])

    # Merge all IFO columns
    ifo_cols = [col for col in ifo_data.columns if col.startswith('ifo_')]
    if not ifo_cols:
        raise ValueError("ifo_data must contain at least one column starting with 'ifo_'")

    df = pd.merge(df, ifo_data[['Year-Month'] + ifo_cols], on='Year-Month', how='left')

    # Generate lag, rolling mean, and std features for all IFO columns
    for col in ifo_cols:
        df[f'{col}_lag'] = df.groupby('area_num')[col].shift(1)
        df[f'{col}_ma3'] = df.groupby('area_num')[col].rolling(3).mean().reset_index(level=0, drop=True)
        df[f'{col}_std3'] = df.groupby('area_num')[col].rolling(3).std().reset_index(level=0, drop=True)

    # Add adjusted sales lag/ma
    df['adjusted_sales_lag'] = df.groupby('area_num')['adjusted_sales'].shift(1)
    df['adjusted_sales_ma3'] = df.groupby('area_num')['adjusted_sales'].rolling(3).mean().reset_index(level=0, drop=True)

    # Merge holiday
    df = pd.merge(df, holiday_data, on='Year-Month', how='left')
    df['holiday_effect'] = df['holiday'].fillna(0).astype(int)

    # Time features
    df['lockdown_2020'] = (df['Year-Month'] == pd.to_datetime('2021-01-01')).astype(int)
    df['month_num'] = df['Year-Month'].dt.month
    df['trade_season'] = df['month_num'].apply(lambda x: 1 if x == 11 else (0.75 if x == 10 else 0))
    df['war_ukraine_2022'] = (df['Year-Month'] >= pd.to_datetime('2022-02-01')).astype(int)
    df['time_index'] = df.groupby('area_num').cumcount() + 1
    df['monthly_growth_rate'] = df.groupby('area_num')['adjusted_sales'].pct_change().fillna(0)

    return df

In [ ]:
df_trade=add_exog_factor(df_trade, ifo_trade, holiday_data)
df_service=add_exog_factor(df_service, ifo_service, holiday_data)

In [ ]:
#Forecast IFO
def forecast_ifo(ifo_df: pd.DataFrame, periods: int = 14) -> pd.DataFrame:
    ifo_df = ifo_df.copy()
    ifo_df['Year-Month'] = pd.to_datetime(ifo_df['Year-Month'])
    ifo_df = ifo_df.drop_duplicates(subset='Year-Month', keep='last')
    ifo_df = ifo_df.set_index('Year-Month').asfreq('MS')

    ifo_cols = [col for col in ifo_df.columns if col.startswith('ifo_')]
    if not ifo_cols:
        raise ValueError("ifo_df must contain at least one column starting with 'ifo_'")

    forecast_df = pd.DataFrame(index=pd.date_range(ifo_df.index[-1] + pd.DateOffset(months=1), periods=periods, freq='MS'))

    for col in ifo_cols:
        df = ifo_df.copy()
        df['slope'] = df[col].diff()
        trend_slope = df['slope'].rolling(window=12).mean().iloc[-1]

        df['month'] = df.index.month
        monthly_avg = df.groupby('month')[col].mean()
        overall_avg = df[col].mean()
        seasonal_offsets = monthly_avg - overall_avg

        current_value = df[col].dropna().iloc[-1]
        forecast_values = []

        for dt in forecast_df.index:
            month = dt.month
            seasonal = seasonal_offsets.get(month, 0)
            current_value += trend_slope
            forecast_value = current_value + seasonal
            forecast_values.append(forecast_value)

        forecast_df[f'forecast_{col}'] = forecast_values

    return forecast_df

In [ ]:
forecast_ifo_trade = forecast_ifo(ifo_trade)
forecast_ifo_service = forecast_ifo(ifo_service)

In [ ]:
##seasonal moving average and growth rate
def compute_seasonal_ma3_and_growth(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    df['month'] = df['Year-Month'].dt.month
    df['year'] = df['Year-Month'].dt.year

    df['monthly_growth_rate'] = df.groupby('area_num')['adjusted_sales'].pct_change()
    df['monthly_growth_yoy'] = df.groupby(['area_num', 'month'])['monthly_growth_rate'].diff()

    seasonal_ma3 = (
        df.groupby(['area_num', 'month'])['adjusted_sales']
        .rolling(3).mean().reset_index()
        .groupby(['area_num', 'month'])['adjusted_sales'].mean().reset_index()
        .rename(columns={'adjusted_sales': 'seasonal_ma3'})
    )

    avg_growth_delta = (
        df.groupby(['area_num', 'month'])['monthly_growth_yoy'].mean()
        .reset_index().rename(columns={'monthly_growth_yoy': 'avg_growth_delta'})
    )

    return seasonal_ma3, avg_growth_delta

In [ ]:
seasonal_ma3_trade, avg_growth_delta_trade = compute_seasonal_ma3_and_growth(df_trade)
seasonal_ma3_service, avg_growth_delta_service = compute_seasonal_ma3_and_growth(df_service)

In [ ]:
## Traint test split

def time_series_train_test_split(
    X: pd.DataFrame,
    y: pd.Series,
    date_index: pd.Series,
    test_horizon_months: int = 12
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    split_point = date_index.max() - pd.DateOffset(months=test_horizon_months)
    train_mask = date_index < split_point
    test_mask = date_index >= split_point

    return X[train_mask], X[test_mask], y[train_mask], y[test_mask]

In [ ]:
# XGBoost tranining

def train_xgboost_per_area(
    X_train_dict: Dict[int, pd.DataFrame],
    X_test_dict: Dict[int, pd.DataFrame],
    y_train_dict: Dict[int, pd.Series],
    y_test_dict: Dict[int, pd.Series]
) -> Tuple[Dict[int, dict], Dict[int, XGBRegressor]]:

    area_scores = {}
    best_models = {}

    param_grid = {
        'n_estimators': [100, 150],
        'max_depth': [1, 2],
        'learning_rate': [0.01, 0.05],
        'subsample': [0.7, 1.0],
        'colsample_bytree': [0.3, 0.5, 1.0],
        'reg_alpha': [5.0, 10.0, 15.0],
        'reg_lambda': [5.0, 10.0, 15.0]
    }

    for area_id in sorted(X_train_dict.keys()):
        X_train = X_train_dict[area_id]
        X_test = X_test_dict[area_id]
        y_train = y_train_dict[area_id]
        y_test = y_test_dict[area_id]

        if len(X_train) < 6:
            print(f"Skipping area {area_id}: not enough training data.")
            continue

        tscv = TimeSeriesSplit(n_splits=5)
        model = XGBRegressor(random_state=42, verbosity=0)
        grid = GridSearchCV(model, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
        grid.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)], verbose=0)

        best_model = grid.best_estimator_
        best_models[area_id] = best_model

        y_train_pred = best_model.predict(X_train)
        y_test_pred = best_model.predict(X_test)

        mae = mean_absolute_error(y_test, y_test_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        #rmse = mean_squared_error(y_test, y_test_pred, squared=False)
        r2_train = r2_score(y_train, y_train_pred)
        r2_test = r2_score(y_test, y_test_pred)
        r2_gap = r2_train - r2_test

        area_scores[area_id] = {
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2_test,
            'R2_Train': r2_train,
            'R2_Gap': r2_gap,
            'best_params': grid.best_params_
        }

        print(f"\nArea {area_id}:")
        print("Best Params:", grid.best_params_)
        print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2 Test: {r2_test:.3f}, R2 Train: {r2_train:.3f}, Gap: {r2_gap:.3f}")

        print("Top Feature Importances:")
        importances = best_model.feature_importances_
        importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances})
        print(importance_df.sort_values(by='Importance', ascending=False).head(5).to_string(index=False))

    return area_scores, best_models


In [ ]:
##Forecast the adjusted sales

def forecast_adjusted_sales(
    df: pd.DataFrame,
    areas: List[int],
    future_steps: int,
    best_models: Dict[int, XGBRegressor],
    ifo_forecast: pd.DataFrame,
    seasonal_ma3: pd.DataFrame,
    avg_growth_delta: pd.DataFrame,
    holiday_data: pd.DataFrame,
    feature_cols: List[str]
) -> Dict[int, pd.DataFrame]:

    forecast_features = {}

    for area_num in areas:
        if area_num not in best_models:
            continue

        model = best_models[area_num]
        area_df = df[df['area_num'] == area_num].copy().sort_values('Year-Month')
        last_rows = area_df[['adjusted_sales']].tail(10).copy()
        last_date = area_df['Year-Month'].max()
        last_time = area_df['time_index'].iloc[-1]

        area_forecast = []

        for step in range(future_steps):
            future_date = last_date + pd.DateOffset(months=step + 1)
            time_index = last_time + step + 1
            month = future_date.month
            trade_season = 1 if month == 11 else (0.75 if month == 10 else 0)

            last_sales = last_rows['adjusted_sales'].iloc[-1]
            ma3_recursive = last_rows['adjusted_sales'].tail(3).mean()

            ma3_seasonal = seasonal_ma3[
                (seasonal_ma3['area_num'] == area_num) & (seasonal_ma3['month'] == month)
            ]['seasonal_ma3'].values
            ma3_seasonal = ma3_seasonal[0] if len(ma3_seasonal) else ma3_recursive
            ma3 = 0.25 * ma3_recursive + 0.75 * ma3_seasonal

            past_date = future_date - pd.DateOffset(years=1)
            growth_last_year = df[
                (df['area_num'] == area_num) & (df['Year-Month'] == past_date)
            ]['monthly_growth_rate'].values
            delta = avg_growth_delta[
                (avg_growth_delta['area_num'] == area_num) & (avg_growth_delta['month'] == month)
            ]['avg_growth_delta'].values
            growth = growth_last_year[0] + delta[0] if len(growth_last_year) and len(delta) else 0
            growth = np.clip(growth, -0.2, 0.2)

            holiday = holiday_data[holiday_data['Year-Month'] == future_date]['holiday'].values
            holiday_effect = holiday[0] if len(holiday) else 0

            features = {
                'adjusted_sales_lag': last_sales,
                'adjusted_sales_ma3': ma3,
                'holiday_effect': holiday_effect,
                'time_index': time_index,
                'monthly_growth_rate': growth,
                'month_num': month,
                'trade_season': trade_season,
                'war_ukraine_2022': 1
            }

            # Add IFO values if in feature_cols
            for col in ifo_forecast.columns:
                colname = col.replace('forecast_', '')
                if colname in feature_cols:
                    features[colname] = ifo_forecast.loc[future_date, col] if future_date in ifo_forecast.index else ifo_forecast[col].iloc[-1]

            model_features = model.get_booster().feature_names
            for f in model_features:
                if f not in features:
                    features[f] = 0

            X_pred = pd.DataFrame([features])[model_features]

            pred = model.predict(X_pred)[0]
            pred = max(pred, last_sales * 0.95)
            pred = np.clip(pred, ma3_seasonal * 0.9, ma3_seasonal * 1.1)

            features.update({'adjusted_sales': pred, 'Year-Month': future_date, 'area_num': area_num})
            area_forecast.append(features)
            last_rows = pd.concat([last_rows, pd.DataFrame([{'adjusted_sales': pred}])], ignore_index=True)

        forecast_features[area_num] = pd.DataFrame(area_forecast)

    return forecast_features


In [ ]:
##Forecast plotting

def merge_and_plot_forecasts(
    historical_df: pd.DataFrame,
    forecast_df_dict: Dict[int, pd.DataFrame],
    area_col: str = 'area_num',
    date_col: str = 'Year-Month',
    value_col: str = 'adjusted_sales',
    forecast_flag_col: str = 'is_forecast'
):
  
    area_mapping = {1: 'Cologne', 2: 'West', 3: 'East'}

    forecast_df = pd.concat(forecast_df_dict.values(), ignore_index=True)
    forecast_df[forecast_flag_col] = 1
    historical_df = historical_df.copy()
    historical_df[forecast_flag_col] = 0

    combined_df = pd.concat([historical_df, forecast_df], ignore_index=True)
    combined_df[date_col] = pd.to_datetime(combined_df[date_col])
    combined_df = combined_df.sort_values(by=[area_col, date_col]).reset_index(drop=True)

    # Only plot area 1, 2, 3
    plot_areas = [1, 2, 3]
    unique_areas = [a for a in sorted(combined_df[area_col].unique()) if a in plot_areas]
    colors = cm.get_cmap('tab10', len(unique_areas))

    plt.figure(figsize=(14, 7))
    for idx, area_id in enumerate(unique_areas):
        area_data = combined_df[combined_df[area_col] == area_id]
        name = area_mapping.get(area_id, f'Area {area_id}')

        history = area_data[area_data[forecast_flag_col] == 0]
        forecast = area_data[area_data[forecast_flag_col] == 1]

        plt.plot(history[date_col], history[value_col], label=f'{name} - Actual', color=colors(idx))
        plt.plot(forecast[date_col], forecast[value_col], label=f'{name} - Forecast', linestyle='--', color=colors(idx))

    plt.title('Adjusted Sales: Historical vs Forecast (Areas 1-3)')
    plt.xlabel('Year-Month')
    plt.ylabel('Adjusted Sales')
    plt.xticks(rotation=45)
    plt.legend(title="Legend", loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0.0)
    plt.tight_layout()
    plt.show()

    return combined_df


# XGboost Forecasting #

## Trade ##

In [ ]:
#Define target and features---
y = df_trade['adjusted_sales']

#encode number to the 'area' column
#area_mapping = {'Cologne': 1, 'West': 2, 'East': 3,'National/International': 4}
#df_trade['area'] = df_trade['area'].map(area_mapping)

X = df_trade[['ifo_trade','ifo_trade_ma3','adjusted_sales_lag', 'holiday_effect', 'adjusted_sales_ma3', 'time_index', 'monthly_growth_rate', 'month_num', 'area_num', 'trade_season', 'war_ukraine_2022']]

In [ ]:
#train test split
X_train, X_test, y_train, y_test = time_series_train_test_split(X, y, df_trade['Year-Month'])


In [ ]:
df_trade.head()

In [ ]:

# Prepare train/test split dictionaries for each area for df_trade
areas = [1, 2, 3]
X = df_trade[['ifo_trade','ifo_trade_ma3','adjusted_sales_lag', 'holiday_effect', 'adjusted_sales_ma3', 'time_index', 'monthly_growth_rate', 'month_num', 'area_num', 'trade_season', 'war_ukraine_2022']]
y = df_trade['adjusted_sales']
date_index = df_trade['Year-Month']

X_train_dict = {}
X_test_dict = {}
y_train_dict = {}
y_test_dict = {}

for area in areas:
    mask = df_trade['area_num'] == area
    X_area = X[mask]
    y_area = y[mask]
    date_area = date_index[mask]
    split_point = date_area.max() - pd.DateOffset(months=12)
    train_mask = date_area < split_point
    test_mask = date_area >= split_point
    X_train_dict[area] = X_area[train_mask]
    X_test_dict[area] = X_area[test_mask]
    y_train_dict[area] = y_area[train_mask]
    y_test_dict[area] = y_area[test_mask]

area_scores_monthly, best_models_trade = train_xgboost_per_area(
    X_train_dict,
    X_test_dict,
    y_train_dict,
    y_test_dict
)

In [ ]:
#forecast until mid-2026
forecast_df_dict_trade = forecast_adjusted_sales(
    df=df_trade,
    areas=df_trade['area_num'].unique().tolist(),
    future_steps=14,
    best_models=best_models_trade,
    ifo_forecast=forecast_ifo_trade,
    seasonal_ma3=seasonal_ma3_trade,
    avg_growth_delta=avg_growth_delta_trade,
    holiday_data=holiday_data,
    feature_cols=[
        'ifo_trade',
        'adjusted_sales_lag', 'adjusted_sales_ma3',
        'holiday_effect', 'time_index',
        'monthly_growth_rate', 'month_num',
        'trade_season', 'war_ukraine_2022'
    ]
)


In [ ]:
#trade forecast by area
combined_df_trade = merge_and_plot_forecasts(df_trade, forecast_df_dict_trade)

## Service

In [ ]:
#Define target and features---
y = df_service['adjusted_sales']

#filter area_num = 1,2,3
df_service = df_service[df_service['area_num'].isin([1, 2, 3])].copy()

X = df_service[['ifo_service','ifo_service_ma3','adjusted_sales_lag', 'holiday_effect', 'adjusted_sales_ma3', 'time_index', 'monthly_growth_rate', 'month_num', 'area_num', 'war_ukraine_2022']]

In [ ]:
#train test split
X_train, X_test, y_train, y_test = time_series_train_test_split(X, y, df_service['Year-Month'])

In [ ]:
# Prepare train/test split dictionaries for each area
areas = [1, 2, 3]
X = df_service[['ifo_service','ifo_service_ma3','adjusted_sales_lag', 'holiday_effect', 'adjusted_sales_ma3', 'time_index', 'monthly_growth_rate', 'month_num', 'area_num', 'war_ukraine_2022']]
y = df_service['adjusted_sales']
date_index = df_service['Year-Month']

X_train_dict = {}
X_test_dict = {}
y_train_dict = {}
y_test_dict = {}

for area in areas:
    mask = df_service['area_num'] == area
    X_area = X[mask]
    y_area = y[mask]
    date_area = date_index[mask]
    split_point = date_area.max() - pd.DateOffset(months=12)
    train_mask = date_area < split_point
    test_mask = date_area >= split_point
    X_train_dict[area] = X_area[train_mask]
    X_test_dict[area] = X_area[test_mask]
    y_train_dict[area] = y_area[train_mask]
    y_test_dict[area] = y_area[test_mask]

area_scores_service, best_models_service = train_xgboost_per_area(
    X_train_dict,
    X_test_dict,
    y_train_dict,
    y_test_dict
)

In [ ]:
forecast_df_dict_service = forecast_adjusted_sales(
    df=df_service,
    areas=df_service['area_num'].unique().tolist(),
    future_steps=14,
    best_models=best_models_service,
    ifo_forecast=forecast_ifo_service,
    seasonal_ma3=seasonal_ma3_service,
    avg_growth_delta=avg_growth_delta_service,
    holiday_data=holiday_data,
    feature_cols=[['ifo_service','ifo_service_ma3','adjusted_sales_lag', 'holiday_effect', 'adjusted_sales_ma3', 'time_index', 'monthly_growth_rate', 'month_num', 'area_num', 'war_ukraine_2022']]
)


In [ ]:
combined_df_service = merge_and_plot_forecasts(df_service, forecast_df_dict_service)

# Deep Dive - Analysis # 

For the industry segments "Services" and "Trade", how does total adjusted sales change if the number of transactions changes by ±1%, 5%, or 10% or number of customers changes by ±10,50,100?

In [ ]:
# Step 1: Filter for Services, Trade, Digital, and Year 2024
df_focus = df_clustered[
    (df_clustered['Customer_industry_name'].isin(['Services', 'Trade'])) &
    (df_clustered['Product_group'] == 2.0) &
    (df_clustered['Year-Month'].str.startswith('2024'))
]

# Step 2: Aggregate monthly baselines by Year-Month and Industry
df_monthly_base = df_focus.groupby(['Year-Month', 'Customer_industry_name']).agg({
    'Sum_Adjusted_Sales': 'sum',
    'customer number Industry': 'sum',
    'Transactions': 'sum'
}).reset_index()

# Step 3: Calculate ratios
df_monthly_base['sales_per_transaction'] = df_monthly_base['Sum_Adjusted_Sales'] / df_monthly_base['Transactions']
df_monthly_base['sales_per_customer'] = df_monthly_base['Sum_Adjusted_Sales'] / df_monthly_base['customer number Industry']

# Step 4: Simulate impact
transaction_percents = [1, 5, 10]
customer_changes = [10, 50, 100]
results = []

for _, row in df_monthly_base.iterrows():
    ym = row['Year-Month']
    industry = row['Customer_industry_name']
    
    base_sales = row['Sum_Adjusted_Sales']
    base_customers = row['customer number Industry']
    base_transactions = row['Transactions']
    spc = row['sales_per_customer']
    spt = row['sales_per_transaction']

    results.append({
        'Year-Month': ym,
        'Customer_industry': industry,
        'Scenario': 'Baseline',
        'Estimated_Sales': round(base_sales, 2)
    })

    # Absolute customer changes
    for delta in customer_changes:
        for direction in [-1, 1]:
            change = direction * delta
            sign = "+" if direction > 0 else "-"
            est_customers = base_customers + change
            est_sales = est_customers * spc
            results.append({
                'Year-Month': ym,
                'Customer_industry': industry,
                'Scenario': f'Customers {sign}{abs(change)}',
                'Estimated_Sales': round(est_sales, 2)
            })

    # Percent transaction changes
    for pct in transaction_percents:
        for direction in [-1, 1]:
            factor = 1 + (direction * pct / 100)
            sign = "+" if direction > 0 else "-"
            est_sales = base_transactions * factor * spt
            results.append({
                'Year-Month': ym,
                'Customer_industry': industry,
                'Scenario': f'Transactions {sign}{pct}%',
                'Estimated_Sales': round(est_sales, 2)
            })


# Step 5: Create final table
df_monthly_impact = pd.DataFrame(results)

df_monthly_impact['Year-Month'] = pd.to_datetime(df_monthly_impact['Year-Month']) + pd.DateOffset(years=1)
df_monthly_impact['Year-Month'] = df_monthly_impact['Year-Month'].dt.strftime('%Y-%m')

In [ ]:
df_monthly_impact.head(10)


## Scenario Dashbard ##

In [ ]:
# --- Prepare data ---
df_filtered = df_monthly_impact.copy()
df_filtered['Year-Month'] = pd.to_datetime(df_filtered['Year-Month'])
df_filtered = df_filtered[df_filtered['Year-Month'].dt.year == 2025]

# Extract Baseline for subtraction
baseline_df = df_filtered[df_filtered['Scenario'] == 'Baseline']
baseline_map = baseline_df.set_index(['Year-Month', 'Customer_industry'])['Estimated_Sales'].to_dict()

# Add delta to all rows
df_filtered['Baseline'] = df_filtered.apply(
    lambda row: baseline_map.get((row['Year-Month'], row['Customer_industry']), 0),
    axis=1
)
df_filtered['Delta'] = df_filtered['Estimated_Sales'] - df_filtered['Baseline']

# Extract controls
industries = df_filtered['Customer_industry'].unique()
directions = ['Increase', 'Decrease']
scenario_types = ['Transactions', 'Customers']
transaction_pct = [1, 5, 10]
customer_abs = [10, 50, 100]

# --- Dash App ---
app = JupyterDash(__name__)
app.layout = html.Div([
    html.H3("Scenario Analysis: Estimated Impact on Adjusted Sales (2025)",
        style={
            'color': 'lightblue',
            'textAlign': 'center',
            'fontWeight': 'bold',
            'fontSize': '28px'
        }
    ),
    
    html.Div([
        html.Label("Industry"),
        dcc.Dropdown(id='industry-dropdown', options=[{'label': i, 'value': i} for i in industries], value=industries[0])
    ], style={'width': '24%', 'display': 'inline-block'}),

    html.Div([
        html.Label("Change Type"),
        dcc.Dropdown(id='change-type-dropdown', options=[{'label': i, 'value': i} for i in scenario_types], value='Transactions')
    ], style={'width': '24%', 'display': 'inline-block', 'marginLeft': '1%'}),

    html.Div([
        html.Label("Direction"),
        dcc.Dropdown(id='direction-dropdown', options=[{'label': i, 'value': i} for i in directions], value='Increase')
    ], style={'width': '24%', 'display': 'inline-block', 'marginLeft': '1%'}),

    html.Div([
        html.Label("Change Amount"),
        dcc.Dropdown(id='change-amount-dropdown')
    ], style={'width': '24%', 'display': 'inline-block', 'marginLeft': '1%'}),

    dcc.Graph(id='sales-impact-line')
])

# --- Callback for dynamic dropdown ---
@app.callback(
    Output('change-amount-dropdown', 'options'),
    Output('change-amount-dropdown', 'value'),
    Input('change-type-dropdown', 'value')
)
def update_amount_options(change_type):
    if change_type == 'Transactions':
        options = [{'label': f'{p}%', 'value': f'{p}%'} for p in transaction_pct]
        value = '1%'
    else:
        options = [{'label': f'{p}', 'value': f'{p}'} for p in customer_abs]
        value = '10'
    return options, value

# --- Callback to update chart ---
@app.callback(
    Output('sales-impact-line', 'figure'),
    Input('industry-dropdown', 'value'),
    Input('change-type-dropdown', 'value'),
    Input('direction-dropdown', 'value'),
    Input('change-amount-dropdown', 'value')
)
def update_chart(industry, change_type, direction, amount):
    if change_type == 'Transactions':
        scenario_label = f'Transactions {"+" if direction == "Increase" else "-"}{amount}'
    else:
        scenario_label = f'Customers {"+" if direction == "Increase" else "-"}{amount}'

    filtered = df_filtered[
        (df_filtered['Customer_industry'] == industry) &
        (df_filtered['Scenario'] == scenario_label)
    ]

    fig = px.line(
        filtered.sort_values('Year-Month'),
        x='Year-Month',
        y='Delta',
        title=f"{scenario_label} Impact on Sales for {industry} (2025)",
        markers=True
    )
    fig.update_layout(yaxis_title="Δ Estimated Sales (€)", xaxis_title="Month", height=400)
    return fig

# --- Run in notebook ---
app.run(mode='inline', port=8062)
